# 4 часть 

Using two best models (one for classification and one for regression) from H2 or H3 (it can be Linear model or non-linear model it doesn't matter) with original features set and apply  for features selection (`feature importance`, `permutation importance`, `L1 regularization`, `Random variable method`, `Recursive feature elimination`). And create models with the same or better quality with smaller amount of input features

## Step by step guide

1. Fit selected model in this notebook and get train and valid scores and calculate number **input** of features, put this information to the table below (first row in table)
. Next step, for each feature selection algorithm (`feature importance`, `permutation importance`, `L1 regularization`, `Random variable method`, `Recursive feature elimination`)
    - Select one feature selection algorithm 
    - You should start with original model (**with all features and original hyperparameters**), then apply feature selection method and remove insignificant features
    - Fit model with selected features only (without insignificant features) and get scores (train and valid) and put this information to the table, with number of selected features (see table with all details).
    - Select next features selection method and repeat
2. **Combine top (selected) features from classification and regression models** into one list
3. Prepare submission files with **number of top features and prediction columns** ('__price_predict', '__price_num_features', '__churn_prob', '__churn_num_features', '__priority') 



In [1]:
import numpy as np
import pandas as pd

scoring_churn = pd.DataFrame([
    # Classification
    ('For classification model', 'Original features set', 0, 'metric_for_churn', 0., 0.),
    ('For classification model', 'Feature importance', 0, 'metric_for_churn', 0., 0.),
    ('For classification model', 'Permutation importance', 0, 'metric_for_churn', 0., 0.),
    ('For classification model', 'L1 regularization', 0, 'metric_for_churn', 0., 0.),
    ('For classification model', 'Random variable method', 0, 'metric_for_churn', 0., 0.),
    ('For classification model', 'Recursive feature elimination', 0, 'metric_for_churn', 0., 0.),
], columns=['Type of model', 'after use Feature Selection', 'number of features', 'metric_name', 'score_train', 'score_valid'])

scoring_churn

,Type of model,after use Feature Selection,number of features,metric_name,score_train,score_valid
0,For classification model,Original features set,0,metric_for_churn,0.0,0.0
1,For classification model,Feature importance,0,metric_for_churn,0.0,0.0
2,For classification model,Permutation importance,0,metric_for_churn,0.0,0.0
3,For classification model,L1 regularization,0,metric_for_churn,0.0,0.0
4,For classification model,Random variable method,0,metric_for_churn,0.0,0.0
5,For classification model,Recursive feature elimination,0,metric_for_churn,0.0,0.0


In [ ]:
import numpy as np
import pandas as pd

# table with metrics for all models
# 
# ('For regression model', 'Feature importance', 21, 'metric_for_price', 0.1321, 0.1421),

scoring_price = pd.DataFrame([
    # Regression
    ('For regression model', 'Original features set', 0, 'metric_for_price', 0., 0.),
    ('For regression model', 'Feature importance', 0, 'metric_for_price', 0., 0.),
    ('For regression model', 'Permutation importance', 0, 'metric_for_price', 0., 0.),
    ('For regression model', 'L1 regularization', 0, 'metric_for_price', 0., 0.),
    ('For regression model', 'Random variable method', 0, 'metric_for_price', 0., 0.),
    ('For regression model', 'Recursive feature elimination', 0, 'metric_for_price', 0., 0.),
], columns=['Type of model', 'after use Feature Selection', 'number of features', 'metric_name', 'score_train', 'score_valid'])

scoring_price

,Type of model,after use Feature Selection,number of features,metric_name,score_train,score_valid
0,For regression model,Original features set,0,metric_for_price,0.0,0.0
1,For regression model,Feature importance,0,metric_for_price,0.0,0.0
2,For regression model,Permutation importance,0,metric_for_price,0.0,0.0
3,For regression model,L1 regularization,0,metric_for_price,0.0,0.0
4,For regression model,Random variable method,0,metric_for_price,0.0,0.0
5,For regression model,Recursive feature elimination,0,metric_for_price,0.0,0.0


In [5]:
import numpy as np
import pandas as pd
pd.options.display.max_rows = 1000

# Your Name Surname 
# !!! should not change in other hypotheses
NAME = "Kirill_Viksman"

dataset = pd.read_csv(f"1_{NAME}.csv", index_col=0)
SHAPE = dataset.shape

train_all = dataset[dataset['dtype'] == 'train']
test = dataset[dataset['dtype'] == 'test']

SHAPE, train_all.shape, test.shape

((30471, 399), (20483, 399), (9988, 399))

## Metrics for estimate model quality

In [6]:
from sklearn.metrics import mean_squared_log_error
from sklearn.metrics import roc_auc_score

# this metric bigger is better for price and churn target
def metric_for_price(y_true, y_pred):
    """ mean_squared_log_error. bigger is better """
    return round(-mean_squared_log_error(y_true=y_true, y_pred=y_pred), 3)


def metric_for_churn(y_true, y_score):
    """ roc_auc_score. bigger is better """
    return round(roc_auc_score(y_true=y_true, y_score=y_score), 3)

# YOUR WORK STARTS HERE

In [ ]:

# H4: Feature Selection for best H2/H3 models


import json
import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error, roc_auc_score
from sklearn.linear_model import Ridge, Lasso, LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import RFE

warnings.filterwarnings("ignore")
pd.options.display.max_rows = 1000

RANDOM_STATE = 47


#  Safety reset

dataset = dataset.reset_index(drop=True)
train_all = dataset[dataset["dtype"] == "train"].copy().reset_index(drop=True)
test = dataset[dataset["dtype"] == "test"].copy().reset_index(drop=True)


#  Original feature set from H1

target_cols = ["__price", "__churn", "dtype"]
object_cols = dataset.select_dtypes(include=["object"]).columns.tolist()

feature_cols_all = [
    c for c in dataset.columns
    if c not in target_cols and c not in object_cols
]

X_all_full = dataset[feature_cols_all].replace([np.inf, -np.inf], np.nan).reset_index(drop=True)

train_mask = dataset["dtype"].eq("train").values
test_mask = dataset["dtype"].eq("test").values

X_train_full = X_all_full.loc[train_mask].reset_index(drop=True)
X_test_full = X_all_full.loc[test_mask].reset_index(drop=True)

y_price_all = train_all["__price"].reset_index(drop=True)
y_churn_all = train_all["__churn"].astype(int).reset_index(drop=True)

print("Original numeric feature set:", len(feature_cols_all))
print("X_train_full:", X_train_full.shape)
print("X_test_full:", X_test_full.shape)


#  Validation split for price
# OOT split by timestamp if timestamp exists

if "timestamp" in train_all.columns:
    train_tmp = train_all.copy()
    train_tmp["timestamp_dt"] = pd.to_datetime(train_tmp["timestamp"])
    train_sorted = train_tmp.sort_values("timestamp_dt").reset_index()

    price_order = train_sorted["index"].values
    oot_cutoff = train_sorted["timestamp_dt"].quantile(0.80)

    dev_mask_price = train_sorted["timestamp_dt"] < oot_cutoff
    valid_mask_price = train_sorted["timestamp_dt"] >= oot_cutoff

    X_price_sorted = X_train_full.loc[price_order].reset_index(drop=True)

    X_price_dev_full = X_price_sorted.loc[dev_mask_price].reset_index(drop=True)
    X_price_valid_full = X_price_sorted.loc[valid_mask_price].reset_index(drop=True)

    y_price_dev = train_sorted.loc[dev_mask_price, "__price"].reset_index(drop=True)
    y_price_valid = train_sorted.loc[valid_mask_price, "__price"].reset_index(drop=True)
else:
    split_pos = int(len(X_train_full) * 0.8)

    X_price_dev_full = X_train_full.iloc[:split_pos].reset_index(drop=True)
    X_price_valid_full = X_train_full.iloc[split_pos:].reset_index(drop=True)

    y_price_dev = y_price_all.iloc[:split_pos].reset_index(drop=True)
    y_price_valid = y_price_all.iloc[split_pos:].reset_index(drop=True)

print("price dev:", X_price_dev_full.shape)
print("price valid:", X_price_valid_full.shape)


#  Validation split for churn


X_churn_dev_full, X_churn_valid_full, y_churn_dev, y_churn_valid = train_test_split(
    X_train_full,
    y_churn_all,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_churn_all
)

X_churn_dev_full = X_churn_dev_full.reset_index(drop=True)
X_churn_valid_full = X_churn_valid_full.reset_index(drop=True)
y_churn_dev = y_churn_dev.reset_index(drop=True)
y_churn_valid = y_churn_valid.reset_index(drop=True)

print("churn dev:", X_churn_dev_full.shape)
print("churn valid:", X_churn_valid_full.shape)


#  Metrics

def metric_price_raw(y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred), 1e-9, None)
    return -mean_squared_log_error(y_true, y_pred)

def metric_churn_raw(y_true, y_score):
    return roc_auc_score(y_true, y_score)

def metric_price_round(y_true, y_pred):
    return round(metric_price_raw(y_true, y_pred), 3)

def metric_churn_round(y_true, y_score):
    return round(metric_churn_raw(y_true, y_score), 3)

def price_perm_scorer(estimator, X, y):
    pred = estimator.predict(X)
    return metric_price_raw(y, pred)

def churn_perm_scorer(estimator, X, y):
    prob = estimator.predict_proba(X)[:, 1]
    return metric_churn_raw(y, prob)


#  Base models from H2/H3
# Regression: Ridge from H2 linear models.
# Classification: Logistic Regression from H2 linear models.

price_base_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1000.0, random_state=RANDOM_STATE))
])

churn_base_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        C=0.01,
        penalty="l2",
        solver="lbfgs",
        max_iter=3000,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])


#  Helper functions

def fit_predict_regression(features, model=None):
    if model is None:
        model = clone(price_base_model)

    model.fit(X_price_dev_full[features], y_price_dev)

    pred_train = np.clip(model.predict(X_price_dev_full[features]), 1e-9, None)
    pred_valid = np.clip(model.predict(X_price_valid_full[features]), 1e-9, None)

    return model, pred_train, pred_valid

def fit_predict_classification(features, model=None):
    if model is None:
        model = clone(churn_base_model)

    model.fit(X_churn_dev_full[features], y_churn_dev)

    prob_train = model.predict_proba(X_churn_dev_full[features])[:, 1]
    prob_valid = model.predict_proba(X_churn_valid_full[features])[:, 1]

    return model, prob_train, prob_valid

def evaluate_regression(features, label, model=None):
    model, pred_train, pred_valid = fit_predict_regression(features, model)

    return {
        "Type of model": "For regression model",
        "after use Feature Selection": label,
        "number of features": len(features),
        "metric_name": "metric_for_price",
        "score_train": metric_price_round(y_price_dev, pred_train),
        "score_valid": metric_price_round(y_price_valid, pred_valid),
        "raw_score_valid": metric_price_raw(y_price_valid, pred_valid),
        "features": list(features),
        "model": model,
    }

def evaluate_classification(features, label, model=None):
    model, prob_train, prob_valid = fit_predict_classification(features, model)

    return {
        "Type of model": "For classification model",
        "after use Feature Selection": label,
        "number of features": len(features),
        "metric_name": "metric_for_churn",
        "score_train": metric_churn_round(y_churn_dev, prob_train),
        "score_valid": metric_churn_round(y_churn_valid, prob_valid),
        "raw_score_valid": metric_churn_raw(y_churn_valid, prob_valid),
        "features": list(features),
        "model": model,
    }

def coef_importance_from_pipeline(pipe, features):
    coef = pipe.named_steps["model"].coef_
    coef = np.ravel(coef)
    return pd.Series(np.abs(coef), index=features).sort_values(ascending=False)

def choose_top_k_by_validation(task, ranked_features, method_label):
    k_grid = [10, 20, 30, 40, 60, 80, 100, 120, 160, 200, 240, 300, len(ranked_features)]

    candidates = []

    for k in k_grid:
        k = min(k, len(ranked_features))

        if k <= 0:
            continue

        feats = list(ranked_features[:k])

        if task == "price":
            row = evaluate_regression(feats, method_label)
        else:
            row = evaluate_classification(feats, method_label)

        candidates.append(row)

    candidates = sorted(
        candidates,
        key=lambda x: (x["raw_score_valid"], -x["number of features"]),
        reverse=True
    )

    return candidates[0], candidates

def build_table(results, task):
    rows = []

    for row in results:
        rows.append((
            row["Type of model"],
            row["after use Feature Selection"],
            row["number of features"],
            row["metric_name"],
            row["score_train"],
            row["score_valid"],
        ))

    return pd.DataFrame(
        rows,
        columns=[
            "Type of model",
            "after use Feature Selection",
            "number of features",
            "metric_name",
            "score_train",
            "score_valid"
        ]
    )

def choose_final_reduced_model(results):
    baseline = results[0]
    reduced = [r for r in results[1:] if r["number of features"] < baseline["number of features"]]

    same_or_better = [
        r for r in reduced
        if r["score_valid"] >= baseline["score_valid"]
    ]

    if len(same_or_better) > 0:
        selected = sorted(
            same_or_better,
            key=lambda x: (x["score_valid"], x["raw_score_valid"], -x["number of features"]),
            reverse=True
        )[0]
    else:
        selected = sorted(
            reduced,
            key=lambda x: (x["raw_score_valid"], -x["number of features"]),
            reverse=True
        )[0]

    return selected


# PRICE FEATURE SELECTION

price_results = []

# Original features set
price_baseline = evaluate_regression(feature_cols_all, "Original features set")
price_results.append(price_baseline)

print("PRICE baseline:", price_baseline["score_train"], price_baseline["score_valid"], "features:", price_baseline["number of features"])


#  Feature importance
# Linear model coefficients

price_base_fitted = price_baseline["model"]
price_coef_imp = coef_importance_from_pipeline(price_base_fitted, feature_cols_all)

price_fi_row, price_fi_candidates = choose_top_k_by_validation(
    task="price",
    ranked_features=price_coef_imp.index.tolist(),
    method_label="Feature importance"
)

price_results.append(price_fi_row)


#  Permutation importance

perm_sample_size = min(3000, X_price_valid_full.shape[0])

perm_idx = np.random.RandomState(RANDOM_STATE).choice(
    np.arange(X_price_valid_full.shape[0]),
    size=perm_sample_size,
    replace=False
)

price_perm = permutation_importance(
    price_base_fitted,
    X_price_valid_full.iloc[perm_idx][feature_cols_all],
    y_price_valid.iloc[perm_idx],
    scoring=price_perm_scorer,
    n_repeats=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

price_perm_imp = pd.Series(
    price_perm.importances_mean,
    index=feature_cols_all
).sort_values(ascending=False)

price_perm_ranked = price_perm_imp[price_perm_imp > 0].index.tolist()

if len(price_perm_ranked) < 20:
    price_perm_ranked = price_perm_imp.index.tolist()

price_perm_row, price_perm_candidates = choose_top_k_by_validation(
    task="price",
    ranked_features=price_perm_ranked,
    method_label="Permutation importance"
)

price_results.append(price_perm_row)


#  L1 regularization
# Lasso removes weak features by setting coefficients to zero

lasso_alphas = [0.0005, 0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1]
price_l1_candidates = []

for alpha in lasso_alphas:
    lasso_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Lasso(
            alpha=alpha,
            max_iter=7000,
            random_state=RANDOM_STATE
        ))
    ])

    lasso_pipe.fit(X_price_dev_full[feature_cols_all], y_price_dev)

    coef = np.ravel(lasso_pipe.named_steps["model"].coef_)
    selected = [f for f, c in zip(feature_cols_all, coef) if abs(c) > 1e-8]

    if len(selected) == 0:
        continue

    row = evaluate_regression(selected, "L1 regularization")
    row["l1_alpha"] = alpha
    price_l1_candidates.append(row)

if len(price_l1_candidates) == 0:
    price_l1_row = price_fi_row.copy()
    price_l1_row["after use Feature Selection"] = "L1 regularization"
else:
    price_l1_candidates = sorted(
        price_l1_candidates,
        key=lambda x: (x["raw_score_valid"], -x["number of features"]),
        reverse=True
    )
    price_l1_row = price_l1_candidates[0]

price_results.append(price_l1_row)


#  Random variable method


rng_local = np.random.RandomState(RANDOM_STATE)

X_price_dev_random = X_price_dev_full[feature_cols_all].copy()
X_price_valid_random = X_price_valid_full[feature_cols_all].copy()

X_price_dev_random["__random_feature__"] = rng_local.uniform(0, 1, size=len(X_price_dev_random))
X_price_valid_random["__random_feature__"] = rng_local.uniform(0, 1, size=len(X_price_valid_random))

random_features_price = feature_cols_all + ["__random_feature__"]

price_random_model = clone(price_base_model)
price_random_model.fit(X_price_dev_random[random_features_price], y_price_dev)

random_imp = coef_importance_from_pipeline(price_random_model, random_features_price)
random_threshold = random_imp.loc["__random_feature__"]

price_random_selected = random_imp[
    (random_imp > random_threshold) &
    (random_imp.index != "__random_feature__")
].index.tolist()

if len(price_random_selected) < 10:
    price_random_selected = random_imp.drop("__random_feature__").head(60).index.tolist()

price_random_row = evaluate_regression(price_random_selected, "Random variable method")
price_results.append(price_random_row)


#  Recursive feature elimination

price_rfe_candidates = []

for n_select in [20, 30, 40, 60, 80, 100]:
    rfe_estimator = Ridge(alpha=1000.0, random_state=RANDOM_STATE)

    preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    X_rfe_dev = preprocessor.fit_transform(X_price_dev_full[feature_cols_all])

    selector = RFE(
        estimator=rfe_estimator,
        n_features_to_select=n_select,
        step=0.25
    )

    selector.fit(X_rfe_dev, y_price_dev)

    selected = [f for f, keep in zip(feature_cols_all, selector.support_) if keep]

    row = evaluate_regression(selected, "Recursive feature elimination")
    price_rfe_candidates.append(row)

price_rfe_candidates = sorted(
    price_rfe_candidates,
    key=lambda x: (x["raw_score_valid"], -x["number of features"]),
    reverse=True
)

price_rfe_row = price_rfe_candidates[0]
price_results.append(price_rfe_row)

# Final scoring_price table
scoring_price = build_table(price_results, task="price")

best_price_selected = choose_final_reduced_model(price_results)
top_features_price = list(best_price_selected["features"])

print("\nSelected PRICE method:", best_price_selected["after use Feature Selection"])
print("Selected PRICE valid score:", best_price_selected["score_valid"])
print("Selected PRICE raw valid score:", best_price_selected["raw_score_valid"])
print("Selected PRICE number of features:", len(top_features_price))

display(scoring_price)


# CHURN FEATURE SELECTION

churn_results = []

# Original features set
churn_baseline = evaluate_classification(feature_cols_all, "Original features set")
churn_results.append(churn_baseline)

print("\nCHURN baseline:", churn_baseline["score_train"], churn_baseline["score_valid"], "features:", churn_baseline["number of features"])


#  Feature importance
# Logistic regression coefficients

churn_base_fitted = churn_baseline["model"]
churn_coef_imp = coef_importance_from_pipeline(churn_base_fitted, feature_cols_all)

churn_fi_row, churn_fi_candidates = choose_top_k_by_validation(
    task="churn",
    ranked_features=churn_coef_imp.index.tolist(),
    method_label="Feature importance"
)

churn_results.append(churn_fi_row)


#  Permutation importance

perm_sample_size = min(3000, X_churn_valid_full.shape[0])

perm_idx = np.random.RandomState(RANDOM_STATE).choice(
    np.arange(X_churn_valid_full.shape[0]),
    size=perm_sample_size,
    replace=False
)

churn_perm = permutation_importance(
    churn_base_fitted,
    X_churn_valid_full.iloc[perm_idx][feature_cols_all],
    y_churn_valid.iloc[perm_idx],
    scoring=churn_perm_scorer,
    n_repeats=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

churn_perm_imp = pd.Series(
    churn_perm.importances_mean,
    index=feature_cols_all
).sort_values(ascending=False)

churn_perm_ranked = churn_perm_imp[churn_perm_imp > 0].index.tolist()

if len(churn_perm_ranked) < 20:
    churn_perm_ranked = churn_perm_imp.index.tolist()

churn_perm_row, churn_perm_candidates = choose_top_k_by_validation(
    task="churn",
    ranked_features=churn_perm_ranked,
    method_label="Permutation importance"
)

churn_results.append(churn_perm_row)


#  L1 regularization
# LogisticRegression with L1 penalty

l1_C_grid = [0.003, 0.005, 0.01, 0.03, 0.05, 0.1, 0.3]
churn_l1_candidates = []

for C in l1_C_grid:
    l1_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            C=C,
            penalty="l1",
            solver="saga",
            max_iter=5000,
            class_weight="balanced",
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ])

    l1_pipe.fit(X_churn_dev_full[feature_cols_all], y_churn_dev)

    coef = np.ravel(l1_pipe.named_steps["model"].coef_)
    selected = [f for f, c in zip(feature_cols_all, coef) if abs(c) > 1e-8]

    if len(selected) == 0:
        continue

    row = evaluate_classification(selected, "L1 regularization")
    row["l1_C"] = C
    churn_l1_candidates.append(row)

if len(churn_l1_candidates) == 0:
    churn_l1_row = churn_fi_row.copy()
    churn_l1_row["after use Feature Selection"] = "L1 regularization"
else:
    churn_l1_candidates = sorted(
        churn_l1_candidates,
        key=lambda x: (x["raw_score_valid"], -x["number of features"]),
        reverse=True
    )
    churn_l1_row = churn_l1_candidates[0]

churn_results.append(churn_l1_row)


#  Random variable method

rng_local = np.random.RandomState(RANDOM_STATE)

X_churn_dev_random = X_churn_dev_full[feature_cols_all].copy()
X_churn_valid_random = X_churn_valid_full[feature_cols_all].copy()

X_churn_dev_random["__random_feature__"] = rng_local.uniform(0, 1, size=len(X_churn_dev_random))
X_churn_valid_random["__random_feature__"] = rng_local.uniform(0, 1, size=len(X_churn_valid_random))

random_features_churn = feature_cols_all + ["__random_feature__"]

churn_random_model = clone(churn_base_model)
churn_random_model.fit(X_churn_dev_random[random_features_churn], y_churn_dev)

random_imp = coef_importance_from_pipeline(churn_random_model, random_features_churn)
random_threshold = random_imp.loc["__random_feature__"]

churn_random_selected = random_imp[
    (random_imp > random_threshold) &
    (random_imp.index != "__random_feature__")
].index.tolist()

if len(churn_random_selected) < 10:
    churn_random_selected = random_imp.drop("__random_feature__").head(60).index.tolist()

churn_random_row = evaluate_classification(churn_random_selected, "Random variable method")
churn_results.append(churn_random_row)


#  Recursive feature elimination

churn_rfe_candidates = []

for n_select in [20, 30, 40, 60, 80, 100]:
    rfe_estimator = LogisticRegression(
        C=0.01,
        penalty="l2",
        solver="lbfgs",
        max_iter=2000,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE
    )

    preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    X_rfe_dev = preprocessor.fit_transform(X_churn_dev_full[feature_cols_all])

    selector = RFE(
        estimator=rfe_estimator,
        n_features_to_select=n_select,
        step=0.25
    )

    selector.fit(X_rfe_dev, y_churn_dev)

    selected = [f for f, keep in zip(feature_cols_all, selector.support_) if keep]

    row = evaluate_classification(selected, "Recursive feature elimination")
    churn_rfe_candidates.append(row)

churn_rfe_candidates = sorted(
    churn_rfe_candidates,
    key=lambda x: (x["raw_score_valid"], -x["number of features"]),
    reverse=True
)

churn_rfe_row = churn_rfe_candidates[0]
churn_results.append(churn_rfe_row)

# Final scoring_churn table
scoring_churn = build_table(churn_results, task="churn")

best_churn_selected = choose_final_reduced_model(churn_results)
top_features_churn = list(best_churn_selected["features"])

print("\nSelected CHURN method:", best_churn_selected["after use Feature Selection"])
print("Selected CHURN valid score:", best_churn_selected["score_valid"])
print("Selected CHURN raw valid score:", best_churn_selected["raw_score_valid"])
print("Selected CHURN number of features:", len(top_features_churn))

display(scoring_churn)


# Final refit on full train and prediction for full dataset


final_price_model = clone(price_base_model)
final_price_model.fit(X_train_full[top_features_price], y_price_all)

dataset["__price_predict"] = np.clip(
    final_price_model.predict(X_all_full[top_features_price]),
    1e-9,
    None
)

final_churn_model = clone(churn_base_model)
final_churn_model.fit(X_train_full[top_features_churn], y_churn_all)

dataset["__churn_prob"] = final_churn_model.predict_proba(
    X_all_full[top_features_churn]
)[:, 1]

print("\nFINAL PRICE FEATURES:", len(top_features_price))
print("FINAL CHURN FEATURES:", len(top_features_churn))
print(dataset[["__price_predict", "__churn_prob"]].head())

Original numeric feature set: 376
X_train_full: (20483, 376)
X_test_full: (9988, 376)
price dev: (16368, 376)
price valid: (4115, 376)
churn dev: (16386, 376)
churn valid: (4097, 376)
PRICE baseline: -0.154 -0.149 features: 376

Selected PRICE method: Permutation importance
Selected PRICE valid score: -0.146
Selected PRICE raw valid score: -0.146341643960685
Selected PRICE number of features: 160


,Type of model,after use Feature Selection,number of features,metric_name,score_train,score_valid
0,For regression model,Original features set,376,metric_for_price,-0.154,-0.149
1,For regression model,Feature importance,376,metric_for_price,-0.154,-0.149
2,For regression model,Permutation importance,160,metric_for_price,-0.156,-0.146
3,For regression model,L1 regularization,252,metric_for_price,-0.155,-0.150
4,For regression model,Random variable method,296,metric_for_price,-0.154,-0.149
5,For regression model,Recursive feature elimination,60,metric_for_price,-0.157,-0.151



CHURN baseline: 0.971 0.962 features: 376

Selected CHURN method: Permutation importance
Selected CHURN valid score: 0.962
Selected CHURN raw valid score: 0.9624975573207883
Selected CHURN number of features: 120


,Type of model,after use Feature Selection,number of features,metric_name,score_train,score_valid
0,For classification model,Original features set,376,metric_for_churn,0.971,0.962
1,For classification model,Feature importance,200,metric_for_churn,0.971,0.962
2,For classification model,Permutation importance,120,metric_for_churn,0.968,0.962
3,For classification model,L1 regularization,172,metric_for_churn,0.971,0.962
4,For classification model,Random variable method,159,metric_for_churn,0.971,0.962
5,For classification model,Recursive feature elimination,100,metric_for_churn,0.971,0.962



FINAL PRICE FEATURES: 160
FINAL CHURN FEATURES: 120
   __price_predict  __churn_prob
0         3.943595      0.000906
1         7.697608      0.118130
2         1.985606      0.760238
3         2.857207      0.151204
4         4.387671      0.011058


In [ ]:
# save OLD version results before running new version

scoring_price_old = scoring_price.copy()
scoring_churn_old = scoring_churn.copy()

top_features_price_old = list(top_features_price)
top_features_churn_old = list(top_features_churn)

old_price_predict = dataset["__price_predict"].copy()
old_churn_prob = dataset["__churn_prob"].copy()

old_price_num_features = len(top_features_price_old)
old_churn_num_features = len(top_features_churn_old)

print("OLD price features:", old_price_num_features)
print("OLD churn features:", old_churn_num_features)

display(scoring_price_old)
display(scoring_churn_old)

OLD price features: 160
OLD churn features: 120


,Type of model,after use Feature Selection,number of features,metric_name,score_train,score_valid
0,For regression model,Original features set,376,metric_for_price,-0.154,-0.149
1,For regression model,Feature importance,376,metric_for_price,-0.154,-0.149
2,For regression model,Permutation importance,160,metric_for_price,-0.156,-0.146
3,For regression model,L1 regularization,252,metric_for_price,-0.155,-0.150
4,For regression model,Random variable method,296,metric_for_price,-0.154,-0.149
5,For regression model,Recursive feature elimination,60,metric_for_price,-0.157,-0.151


,Type of model,after use Feature Selection,number of features,metric_name,score_train,score_valid
0,For classification model,Original features set,376,metric_for_churn,0.971,0.962
1,For classification model,Feature importance,200,metric_for_churn,0.971,0.962
2,For classification model,Permutation importance,120,metric_for_churn,0.968,0.962
3,For classification model,L1 regularization,172,metric_for_churn,0.971,0.962
4,For classification model,Random variable method,159,metric_for_churn,0.971,0.962
5,For classification model,Recursive feature elimination,100,metric_for_churn,0.971,0.962


In [ ]:

# H4  VERSION
# Feature Selection for price and churn


import json
import warnings
import numpy as np
import pandas as pd


from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error, roc_auc_score
from sklearn.linear_model import Ridge, Lasso, LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import RFE

warnings.filterwarnings("ignore")
pd.options.display.max_rows = 1000

RANDOM_STATE = 47



dataset = pd.read_csv(f"1_{NAME}.csv")

if dataset.columns[0].startswith("Unnamed"):
    dataset = dataset.drop(columns=dataset.columns[0])

generated_cols = [
    "__price_predict",
    "__price_num_features",
    "__churn_prob",
    "__churn_num_features",
    "__priority",
    "price_predict",
    "price_num_features",
    "churn_prob",
    "churn_num_features",
    "priority",
]

dataset = dataset.drop(
    columns=[c for c in generated_cols if c in dataset.columns],
    errors="ignore"
)

dataset = dataset.reset_index(drop=True)

print("Clean dataset loaded:", dataset.shape)





#  Basic checks

print("Starting H4 safe pipeline...")

dataset = dataset.reset_index(drop=True)

train_all = dataset[dataset["dtype"] == "train"].copy().reset_index(drop=True)
test_all = dataset[dataset["dtype"] == "test"].copy().reset_index(drop=True)

print("dataset:", dataset.shape)
print("train:", train_all.shape)
print("test:", test_all.shape)


#  Original numeric feature set from H1

target_cols = ["__price", "__churn", "dtype"]

object_cols = dataset.select_dtypes(include=["object"]).columns.tolist()

feature_cols_all = [
    c for c in dataset.columns
    if c not in target_cols and c not in object_cols
]

X_all_full = dataset[feature_cols_all].replace([np.inf, -np.inf], np.nan).reset_index(drop=True)

train_mask = dataset["dtype"].eq("train").values
test_mask = dataset["dtype"].eq("test").values

X_train_full = X_all_full.loc[train_mask].reset_index(drop=True)
X_test_full = X_all_full.loc[test_mask].reset_index(drop=True)

y_price_all = train_all["__price"].reset_index(drop=True)
y_churn_all = train_all["__churn"].astype(int).reset_index(drop=True)

print("original numeric features:", len(feature_cols_all))


#  Price split: OOT by timestamp 

if "timestamp" in train_all.columns:
    train_tmp = train_all.copy()
    train_tmp["timestamp_dt"] = pd.to_datetime(train_tmp["timestamp"])
    train_sorted = train_tmp.sort_values("timestamp_dt").reset_index()

    price_order = train_sorted["index"].values
    oot_cutoff = train_sorted["timestamp_dt"].quantile(0.80)

    dev_mask_price = train_sorted["timestamp_dt"] < oot_cutoff
    valid_mask_price = train_sorted["timestamp_dt"] >= oot_cutoff

    X_price_sorted = X_train_full.loc[price_order].reset_index(drop=True)

    X_price_dev_full = X_price_sorted.loc[dev_mask_price].reset_index(drop=True)
    X_price_valid_full = X_price_sorted.loc[valid_mask_price].reset_index(drop=True)

    y_price_dev = train_sorted.loc[dev_mask_price, "__price"].reset_index(drop=True)
    y_price_valid = train_sorted.loc[valid_mask_price, "__price"].reset_index(drop=True)
else:
    split_pos = int(len(X_train_full) * 0.8)

    X_price_dev_full = X_train_full.iloc[:split_pos].reset_index(drop=True)
    X_price_valid_full = X_train_full.iloc[split_pos:].reset_index(drop=True)

    y_price_dev = y_price_all.iloc[:split_pos].reset_index(drop=True)
    y_price_valid = y_price_all.iloc[split_pos:].reset_index(drop=True)

print("price dev:", X_price_dev_full.shape)
print("price valid:", X_price_valid_full.shape)


#  Churn split: stratified random split

X_churn_dev_full, X_churn_valid_full, y_churn_dev, y_churn_valid = train_test_split(
    X_train_full,
    y_churn_all,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_churn_all
)

X_churn_dev_full = X_churn_dev_full.reset_index(drop=True)
X_churn_valid_full = X_churn_valid_full.reset_index(drop=True)
y_churn_dev = y_churn_dev.reset_index(drop=True)
y_churn_valid = y_churn_valid.reset_index(drop=True)

print("churn dev:", X_churn_dev_full.shape)
print("churn valid:", X_churn_valid_full.shape)


#  Metrics

def metric_price_raw(y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred), 1e-9, None)
    return -mean_squared_log_error(y_true, y_pred)

def metric_churn_raw(y_true, y_score):
    return roc_auc_score(y_true, y_score)

def metric_price_round(y_true, y_pred):
    return round(metric_price_raw(y_true, y_pred), 3)

def metric_churn_round(y_true, y_score):
    return round(metric_churn_raw(y_true, y_score), 3)

def price_perm_scorer(estimator, X, y):
    pred = estimator.predict(X)
    return metric_price_raw(y, pred)

def churn_perm_scorer(estimator, X, y):
    prob = estimator.predict_proba(X)[:, 1]
    return metric_churn_raw(y, prob)


#  Base models

price_base_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1000.0))
])

churn_base_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        C=0.01,
        penalty="l2",
        solver="lbfgs",
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])


#  Helper functions

def fit_predict_price(features, model=None):
    if model is None:
        model = clone(price_base_model)

    model.fit(X_price_dev_full[features], y_price_dev)

    pred_train = np.clip(model.predict(X_price_dev_full[features]), 1e-9, None)
    pred_valid = np.clip(model.predict(X_price_valid_full[features]), 1e-9, None)

    return model, pred_train, pred_valid

def fit_predict_churn(features, model=None):
    if model is None:
        model = clone(churn_base_model)

    model.fit(X_churn_dev_full[features], y_churn_dev)

    prob_train = model.predict_proba(X_churn_dev_full[features])[:, 1]
    prob_valid = model.predict_proba(X_churn_valid_full[features])[:, 1]

    return model, prob_train, prob_valid

def evaluate_price(features, label, model=None):
    model, pred_train, pred_valid = fit_predict_price(features, model)

    return {
        "Type of model": "For regression model",
        "after use Feature Selection": label,
        "number of features": len(features),
        "metric_name": "metric_for_price",
        "score_train": metric_price_round(y_price_dev, pred_train),
        "score_valid": metric_price_round(y_price_valid, pred_valid),
        "raw_score_valid": metric_price_raw(y_price_valid, pred_valid),
        "features": list(features),
        "model": model,
    }

def evaluate_churn(features, label, model=None):
    model, prob_train, prob_valid = fit_predict_churn(features, model)

    return {
        "Type of model": "For classification model",
        "after use Feature Selection": label,
        "number of features": len(features),
        "metric_name": "metric_for_churn",
        "score_train": metric_churn_round(y_churn_dev, prob_train),
        "score_valid": metric_churn_round(y_churn_valid, prob_valid),
        "raw_score_valid": metric_churn_raw(y_churn_valid, prob_valid),
        "features": list(features),
        "model": model,
    }

def coef_importance_from_pipeline(pipe, features):
    coef = pipe.named_steps["model"].coef_
    coef = np.ravel(coef)
    return pd.Series(np.abs(coef), index=features).sort_values(ascending=False)

def select_best_by_top_k(task, ranked_features, method_label):
    # safe top-k grid, not too heavy
    k_grid = [10, 20, 30, 40, 60, 80, 100, 120, 160, len(ranked_features)]
    candidates = []

    for k in k_grid:
        k = min(k, len(ranked_features))
        if k <= 0:
            continue

        feats = list(ranked_features[:k])

        if task == "price":
            row = evaluate_price(feats, method_label)
        else:
            row = evaluate_churn(feats, method_label)

        candidates.append(row)

    candidates = sorted(
        candidates,
        key=lambda x: (x["raw_score_valid"], -x["number of features"]),
        reverse=True
    )

    return candidates[0], candidates

def make_scoring_table(results):
    rows = []

    for r in results:
        rows.append([
            r["Type of model"],
            r["after use Feature Selection"],
            r["number of features"],
            r["metric_name"],
            r["score_train"],
            r["score_valid"],
        ])

    return pd.DataFrame(
        rows,
        columns=[
            "Type of model",
            "after use Feature Selection",
            "number of features",
            "metric_name",
            "score_train",
            "score_valid",
        ]
    )

def choose_final_model(results):
    baseline = results[0]
    reduced = [r for r in results[1:] if r["number of features"] < baseline["number of features"]]

    # first try: same rounded quality or better with fewer features
    same_or_better = [
        r for r in reduced
        if r["score_valid"] >= baseline["score_valid"]
    ]

    if len(same_or_better) > 0:
        return sorted(
            same_or_better,
            key=lambda x: (x["score_valid"], x["raw_score_valid"], -x["number of features"]),
            reverse=True
        )[0]

    # fallback: best raw score among reduced models
    return sorted(
        reduced,
        key=lambda x: (x["raw_score_valid"], -x["number of features"]),
        reverse=True
    )[0]


# PRICE FEATURE SELECTION

print("\n=== PRICE FEATURE SELECTION ===")

price_results = []

# Baseline
price_baseline = evaluate_price(feature_cols_all, "Original features set")
price_results.append(price_baseline)

print("PRICE baseline:",
      price_baseline["score_train"],
      price_baseline["score_valid"],
      "features:",
      price_baseline["number of features"])

#  Feature importance
price_coef_imp = coef_importance_from_pipeline(price_baseline["model"], feature_cols_all)

price_fi_row, _ = select_best_by_top_k(
    task="price",
    ranked_features=price_coef_imp.index.tolist(),
    method_label="Feature importance"
)
price_results.append(price_fi_row)
print("PRICE feature importance done")

# 2. Permutation importance
perm_sample_size = min(1500, X_price_valid_full.shape[0])
perm_idx = np.random.RandomState(RANDOM_STATE).choice(
    np.arange(X_price_valid_full.shape[0]),
    size=perm_sample_size,
    replace=False
)

price_perm = permutation_importance(
    price_baseline["model"],
    X_price_valid_full.iloc[perm_idx][feature_cols_all],
    y_price_valid.iloc[perm_idx],
    scoring=price_perm_scorer,
    n_repeats=1,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

price_perm_imp = pd.Series(
    price_perm.importances_mean,
    index=feature_cols_all
).sort_values(ascending=False)

price_perm_ranked = price_perm_imp[price_perm_imp > 0].index.tolist()
if len(price_perm_ranked) < 20:
    price_perm_ranked = price_perm_imp.index.tolist()

price_perm_row, _ = select_best_by_top_k(
    task="price",
    ranked_features=price_perm_ranked,
    method_label="Permutation importance"
)
price_results.append(price_perm_row)
print("PRICE permutation importance done")

#  L1 regularization
price_l1_candidates = []
lasso_alphas = [0.0005, 0.001, 0.003, 0.005, 0.01, 0.03, 0.05]

for alpha in lasso_alphas:
    lasso_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Lasso(alpha=alpha, max_iter=4000, random_state=RANDOM_STATE))
    ])

    lasso_pipe.fit(X_price_dev_full[feature_cols_all], y_price_dev)

    coef = np.ravel(lasso_pipe.named_steps["model"].coef_)
    selected = [f for f, c in zip(feature_cols_all, coef) if abs(c) > 1e-8]

    if len(selected) > 0:
        row = evaluate_price(selected, "L1 regularization")
        row["alpha"] = alpha
        price_l1_candidates.append(row)

if len(price_l1_candidates) == 0:
    price_l1_row = price_fi_row.copy()
    price_l1_row["after use Feature Selection"] = "L1 regularization"
else:
    price_l1_row = sorted(
        price_l1_candidates,
        key=lambda x: (x["raw_score_valid"], -x["number of features"]),
        reverse=True
    )[0]

price_results.append(price_l1_row)
print("PRICE L1 regularization done")

#  Random variable method
rng_local = np.random.RandomState(RANDOM_STATE)

X_price_dev_random = X_price_dev_full[feature_cols_all].copy()
X_price_valid_random = X_price_valid_full[feature_cols_all].copy()

X_price_dev_random["__random_feature__"] = rng_local.uniform(0, 1, size=len(X_price_dev_random))
X_price_valid_random["__random_feature__"] = rng_local.uniform(0, 1, size=len(X_price_valid_random))

random_features_price = feature_cols_all + ["__random_feature__"]

price_random_model = clone(price_base_model)
price_random_model.fit(X_price_dev_random[random_features_price], y_price_dev)

price_random_imp = coef_importance_from_pipeline(price_random_model, random_features_price)
price_random_threshold = price_random_imp.loc["__random_feature__"]

price_random_selected = price_random_imp[
    (price_random_imp > price_random_threshold) &
    (price_random_imp.index != "__random_feature__")
].index.tolist()

if len(price_random_selected) < 10:
    price_random_selected = price_random_imp.drop("__random_feature__").head(60).index.tolist()

price_random_row = evaluate_price(price_random_selected, "Random variable method")
price_results.append(price_random_row)
print("PRICE random variable method done")

#  RFE
price_rfe_candidates = []

# reduced grid 
for n_select in [20, 40, 60, 100]:
    rfe_preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    X_price_rfe_dev = rfe_preprocessor.fit_transform(X_price_dev_full[feature_cols_all])

    rfe_selector = RFE(
        estimator=Ridge(alpha=1000.0),
        n_features_to_select=n_select,
        step=0.35
    )

    rfe_selector.fit(X_price_rfe_dev, y_price_dev)

    selected = [f for f, keep in zip(feature_cols_all, rfe_selector.support_) if keep]

    row = evaluate_price(selected, "Recursive feature elimination")
    price_rfe_candidates.append(row)

price_rfe_row = sorted(
    price_rfe_candidates,
    key=lambda x: (x["raw_score_valid"], -x["number of features"]),
    reverse=True
)[0]

price_results.append(price_rfe_row)
print("PRICE RFE done")

scoring_price = make_scoring_table(price_results)

best_price_selected = choose_final_model(price_results)
top_features_price = list(best_price_selected["features"])

print("\nSelected PRICE method:", best_price_selected["after use Feature Selection"])
print("Selected PRICE score_valid:", best_price_selected["score_valid"])
print("Selected PRICE raw_score_valid:", best_price_selected["raw_score_valid"])
print("Selected PRICE number of features:", len(top_features_price))

display(scoring_price)


# CHURN FEATURE SELECTION

print("\n=== CHURN FEATURE SELECTION ===")

churn_results = []

# Baseline
churn_baseline = evaluate_churn(feature_cols_all, "Original features set")
churn_results.append(churn_baseline)

print("CHURN baseline:",
      churn_baseline["score_train"],
      churn_baseline["score_valid"],
      "features:",
      churn_baseline["number of features"])

#  Feature importance
churn_coef_imp = coef_importance_from_pipeline(churn_baseline["model"], feature_cols_all)

churn_fi_row, _ = select_best_by_top_k(
    task="churn",
    ranked_features=churn_coef_imp.index.tolist(),
    method_label="Feature importance"
)
churn_results.append(churn_fi_row)
print("CHURN feature importance done")

#  Permutation importance
perm_sample_size = min(1500, X_churn_valid_full.shape[0])
perm_idx = np.random.RandomState(RANDOM_STATE).choice(
    np.arange(X_churn_valid_full.shape[0]),
    size=perm_sample_size,
    replace=False
)

churn_perm = permutation_importance(
    churn_baseline["model"],
    X_churn_valid_full.iloc[perm_idx][feature_cols_all],
    y_churn_valid.iloc[perm_idx],
    scoring=churn_perm_scorer,
    n_repeats=1,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

churn_perm_imp = pd.Series(
    churn_perm.importances_mean,
    index=feature_cols_all
).sort_values(ascending=False)

churn_perm_ranked = churn_perm_imp[churn_perm_imp > 0].index.tolist()
if len(churn_perm_ranked) < 20:
    churn_perm_ranked = churn_perm_imp.index.tolist()

churn_perm_row, _ = select_best_by_top_k(
    task="churn",
    ranked_features=churn_perm_ranked,
    method_label="Permutation importance"
)
churn_results.append(churn_perm_row)
print("CHURN permutation importance done")

#  L1 regularization
churn_l1_candidates = []
l1_C_grid = [0.003, 0.005, 0.01, 0.03, 0.05, 0.1]

for C in l1_C_grid:
    l1_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            C=C,
            penalty="l1",
            solver="saga",
            max_iter=2500,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])

    l1_pipe.fit(X_churn_dev_full[feature_cols_all], y_churn_dev)

    coef = np.ravel(l1_pipe.named_steps["model"].coef_)
    selected = [f for f, c in zip(feature_cols_all, coef) if abs(c) > 1e-8]

    if len(selected) > 0:
        row = evaluate_churn(selected, "L1 regularization")
        row["C"] = C
        churn_l1_candidates.append(row)

if len(churn_l1_candidates) == 0:
    churn_l1_row = churn_fi_row.copy()
    churn_l1_row["after use Feature Selection"] = "L1 regularization"
else:
    churn_l1_row = sorted(
        churn_l1_candidates,
        key=lambda x: (x["raw_score_valid"], -x["number of features"]),
        reverse=True
    )[0]

churn_results.append(churn_l1_row)
print("CHURN L1 regularization done")

#  Random variable method
rng_local = np.random.RandomState(RANDOM_STATE)

X_churn_dev_random = X_churn_dev_full[feature_cols_all].copy()
X_churn_valid_random = X_churn_valid_full[feature_cols_all].copy()

X_churn_dev_random["__random_feature__"] = rng_local.uniform(0, 1, size=len(X_churn_dev_random))
X_churn_valid_random["__random_feature__"] = rng_local.uniform(0, 1, size=len(X_churn_valid_random))

random_features_churn = feature_cols_all + ["__random_feature__"]

churn_random_model = clone(churn_base_model)
churn_random_model.fit(X_churn_dev_random[random_features_churn], y_churn_dev)

churn_random_imp = coef_importance_from_pipeline(churn_random_model, random_features_churn)
churn_random_threshold = churn_random_imp.loc["__random_feature__"]

churn_random_selected = churn_random_imp[
    (churn_random_imp > churn_random_threshold) &
    (churn_random_imp.index != "__random_feature__")
].index.tolist()

if len(churn_random_selected) < 10:
    churn_random_selected = churn_random_imp.drop("__random_feature__").head(60).index.tolist()

churn_random_row = evaluate_churn(churn_random_selected, "Random variable method")
churn_results.append(churn_random_row)
print("CHURN random variable method done")

#  RFE
churn_rfe_candidates = []

for n_select in [20, 40, 60, 100]:
    rfe_preprocessor = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    X_churn_rfe_dev = rfe_preprocessor.fit_transform(X_churn_dev_full[feature_cols_all])

    rfe_selector = RFE(
        estimator=LogisticRegression(
            C=0.01,
            penalty="l2",
            solver="lbfgs",
            max_iter=1500,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ),
        n_features_to_select=n_select,
        step=0.35
    )

    rfe_selector.fit(X_churn_rfe_dev, y_churn_dev)

    selected = [f for f, keep in zip(feature_cols_all, rfe_selector.support_) if keep]

    row = evaluate_churn(selected, "Recursive feature elimination")
    churn_rfe_candidates.append(row)

churn_rfe_row = sorted(
    churn_rfe_candidates,
    key=lambda x: (x["raw_score_valid"], -x["number of features"]),
    reverse=True
)[0]

churn_results.append(churn_rfe_row)
print("CHURN RFE done")

scoring_churn = make_scoring_table(churn_results)

best_churn_selected = choose_final_model(churn_results)
top_features_churn = list(best_churn_selected["features"])

print("\nSelected CHURN method:", best_churn_selected["after use Feature Selection"])
print("Selected CHURN score_valid:", best_churn_selected["score_valid"])
print("Selected CHURN raw_score_valid:", best_churn_selected["raw_score_valid"])
print("Selected CHURN number of features:", len(top_features_churn))

display(scoring_churn)


# Final refit on full train and prediction for full dataset

final_price_model = clone(price_base_model)
final_price_model.fit(X_train_full[top_features_price], y_price_all)

dataset["__price_predict"] = np.clip(
    final_price_model.predict(X_all_full[top_features_price]),
    1e-9,
    None
)

final_churn_model = clone(churn_base_model)
final_churn_model.fit(X_train_full[top_features_churn], y_churn_all)

dataset["__churn_prob"] = final_churn_model.predict_proba(
    X_all_full[top_features_churn]
)[:, 1]

dataset["__price_num_features"] = len(top_features_price)
dataset["__churn_num_features"] = len(top_features_churn)

print("\nFINAL PRICE FEATURES:", len(top_features_price))
print("FINAL CHURN FEATURES:", len(top_features_churn))
display(dataset[["__price_predict", "__price_num_features", "__churn_prob", "__churn_num_features"]].head())

print("\nH4 safe pipeline finished successfully.")

Clean dataset loaded: (30471, 399)
Starting H4 safe pipeline...
dataset: (30471, 399)
train: (20483, 399)
test: (9988, 399)
original numeric features: 376
price dev: (16368, 376)
price valid: (4115, 376)
churn dev: (16386, 376)
churn valid: (4097, 376)

=== PRICE FEATURE SELECTION ===
PRICE baseline: -0.154 -0.149 features: 376
PRICE feature importance done
PRICE permutation importance done
PRICE L1 regularization done
PRICE random variable method done
PRICE RFE done

Selected PRICE method: Permutation importance
Selected PRICE score_valid: -0.146
Selected PRICE raw_score_valid: -0.14550091632060574
Selected PRICE number of features: 160


,Type of model,after use Feature Selection,number of features,metric_name,score_train,score_valid
0,For regression model,Original features set,376,metric_for_price,-0.154,-0.149
1,For regression model,Feature importance,376,metric_for_price,-0.154,-0.149
2,For regression model,Permutation importance,160,metric_for_price,-0.158,-0.146
3,For regression model,L1 regularization,232,metric_for_price,-0.155,-0.150
4,For regression model,Random variable method,296,metric_for_price,-0.154,-0.149
5,For regression model,Recursive feature elimination,100,metric_for_price,-0.155,-0.151



=== CHURN FEATURE SELECTION ===
CHURN baseline: 0.971 0.962 features: 376
CHURN feature importance done
CHURN permutation importance done
CHURN L1 regularization done
CHURN random variable method done
CHURN RFE done

Selected CHURN method: L1 regularization
Selected CHURN score_valid: 0.962
Selected CHURN raw_score_valid: 0.9619455343321377
Selected CHURN number of features: 172


,Type of model,after use Feature Selection,number of features,metric_name,score_train,score_valid
0,For classification model,Original features set,376,metric_for_churn,0.971,0.962
1,For classification model,Feature importance,376,metric_for_churn,0.971,0.962
2,For classification model,Permutation importance,167,metric_for_churn,0.964,0.957
3,For classification model,L1 regularization,172,metric_for_churn,0.971,0.962
4,For classification model,Random variable method,159,metric_for_churn,0.971,0.962
5,For classification model,Recursive feature elimination,100,metric_for_churn,0.971,0.962



FINAL PRICE FEATURES: 160
FINAL CHURN FEATURES: 172


,__price_predict,__price_num_features,__churn_prob,__churn_num_features
0,4.085270,160,0.001015,172
1,7.212441,160,0.104124,172
2,1.782924,160,0.757906,172
3,3.027526,160,0.095714,172
4,4.252799,160,0.006664,172



H4 safe pipeline finished successfully.


In [14]:
# compare OLD and NEW H4 results

print("OLD PRICE")
print("features:", old_price_num_features)
display(scoring_price_old)

print("NEW PRICE")
print("features:", len(top_features_price))
display(scoring_price)

print("OLD CHURN")
print("features:", old_churn_num_features)
display(scoring_churn_old)

print("NEW CHURN")
print("features:", len(top_features_churn))
display(scoring_churn)

OLD PRICE
features: 160


,Type of model,after use Feature Selection,number of features,metric_name,score_train,score_valid
0,For regression model,Original features set,376,metric_for_price,-0.154,-0.149
1,For regression model,Feature importance,376,metric_for_price,-0.154,-0.149
2,For regression model,Permutation importance,160,metric_for_price,-0.156,-0.146
3,For regression model,L1 regularization,252,metric_for_price,-0.155,-0.150
4,For regression model,Random variable method,296,metric_for_price,-0.154,-0.149
5,For regression model,Recursive feature elimination,60,metric_for_price,-0.157,-0.151


NEW PRICE
features: 160


,Type of model,after use Feature Selection,number of features,metric_name,score_train,score_valid
0,For regression model,Original features set,376,metric_for_price,-0.154,-0.149
1,For regression model,Feature importance,376,metric_for_price,-0.154,-0.149
2,For regression model,Permutation importance,160,metric_for_price,-0.158,-0.146
3,For regression model,L1 regularization,232,metric_for_price,-0.155,-0.150
4,For regression model,Random variable method,296,metric_for_price,-0.154,-0.149
5,For regression model,Recursive feature elimination,100,metric_for_price,-0.155,-0.151


OLD CHURN
features: 120


,Type of model,after use Feature Selection,number of features,metric_name,score_train,score_valid
0,For classification model,Original features set,376,metric_for_churn,0.971,0.962
1,For classification model,Feature importance,200,metric_for_churn,0.971,0.962
2,For classification model,Permutation importance,120,metric_for_churn,0.968,0.962
3,For classification model,L1 regularization,172,metric_for_churn,0.971,0.962
4,For classification model,Random variable method,159,metric_for_churn,0.971,0.962
5,For classification model,Recursive feature elimination,100,metric_for_churn,0.971,0.962


NEW CHURN
features: 172


,Type of model,after use Feature Selection,number of features,metric_name,score_train,score_valid
0,For classification model,Original features set,376,metric_for_churn,0.971,0.962
1,For classification model,Feature importance,376,metric_for_churn,0.971,0.962
2,For classification model,Permutation importance,167,metric_for_churn,0.964,0.957
3,For classification model,L1 regularization,172,metric_for_churn,0.971,0.962
4,For classification model,Random variable method,159,metric_for_churn,0.971,0.962
5,For classification model,Recursive feature elimination,100,metric_for_churn,0.971,0.962


In [15]:
comparison_predictions = pd.DataFrame({
    "old_price_predict": old_price_predict,
    "new_price_predict": dataset["__price_predict"],
    "old_churn_prob": old_churn_prob,
    "new_churn_prob": dataset["__churn_prob"],
})

display(comparison_predictions.describe())

,old_price_predict,new_price_predict,old_churn_prob,new_churn_prob
count,3.047100e+04,3.047100e+04,3.047100e+04,3.047100e+04
mean,6.978108e+00,7.011478e+00,2.463111e-01,2.499273e-01
std,3.245135e+00,3.225187e+00,3.497441e-01,3.538468e-01
min,1.000000e-09,1.000000e-09,1.345407e-07,1.198155e-07
25%,4.897527e+00,4.957520e+00,2.149732e-03,1.906708e-03
50%,6.312982e+00,6.351220e+00,3.008811e-02,2.990166e-02
75%,8.322640e+00,8.354398e+00,4.369580e-01,4.578713e-01
max,2.938843e+01,2.972816e+01,9.998739e-01,9.999311e-01


In [ ]:

# H4 HYBRID 
# price from NEW safe version
# churn from OLD version


from sklearn.base import clone
import numpy as np
import pandas as pd


#  Take PRICE from new  version

top_features_price_hybrid = list(top_features_price)


#  Take CHURN from old version

top_features_churn_hybrid = list(top_features_churn_old)

print("HYBRID PRICE FEATURES:", len(top_features_price_hybrid))
print("HYBRID CHURN FEATURES:", len(top_features_churn_hybrid))

# safety checks
missing_price = [c for c in top_features_price_hybrid if c not in X_train_full.columns]
missing_churn = [c for c in top_features_churn_hybrid if c not in X_train_full.columns]

print("Missing price features:", missing_price)
print("Missing churn features:", missing_churn)

assert len(missing_price) == 0, "Some price features are missing"
assert len(missing_churn) == 0, "Some churn features are missing"


#  Refit final PRICE model on full train

final_price_model = clone(price_base_model)
final_price_model.fit(
    X_train_full[top_features_price_hybrid],
    y_price_all
)

dataset["__price_predict"] = np.clip(
    final_price_model.predict(X_all_full[top_features_price_hybrid]),
    1e-9,
    None
)


# 4. Refit final CHURN model on full train

final_churn_model = clone(churn_base_model)
final_churn_model.fit(
    X_train_full[top_features_churn_hybrid],
    y_churn_all
)

dataset["__churn_prob"] = final_churn_model.predict_proba(
    X_all_full[top_features_churn_hybrid]
)[:, 1]


#  Save number of features

dataset["__price_num_features"] = len(top_features_price_hybrid)
dataset["__churn_num_features"] = len(top_features_churn_hybrid)


#  Update final feature lists for lower H4 cells

top_features_price = list(top_features_price_hybrid)
top_features_churn = list(top_features_churn_hybrid)

# Optional: hybrid scoring tables for display
scoring_price_hybrid = scoring_price.copy()
scoring_churn_hybrid = scoring_churn_old.copy()

print("\nFINAL HYBRID:")
print("price features:", len(top_features_price))
print("churn features:", len(top_features_churn))

display(dataset[
    ["__price_predict", "__price_num_features", "__churn_prob", "__churn_num_features"]
].head())

display(scoring_price_hybrid)
display(scoring_churn_hybrid)

HYBRID PRICE FEATURES: 160
HYBRID CHURN FEATURES: 120
Missing price features: []
Missing churn features: []

FINAL HYBRID:
price features: 160
churn features: 120


,__price_predict,__price_num_features,__churn_prob,__churn_num_features
0,4.085270,160,0.000906,120
1,7.212441,160,0.118130,120
2,1.782924,160,0.760238,120
3,3.027526,160,0.151204,120
4,4.252799,160,0.011058,120


,Type of model,after use Feature Selection,number of features,metric_name,score_train,score_valid
0,For regression model,Original features set,376,metric_for_price,-0.154,-0.149
1,For regression model,Feature importance,376,metric_for_price,-0.154,-0.149
2,For regression model,Permutation importance,160,metric_for_price,-0.158,-0.146
3,For regression model,L1 regularization,232,metric_for_price,-0.155,-0.150
4,For regression model,Random variable method,296,metric_for_price,-0.154,-0.149
5,For regression model,Recursive feature elimination,100,metric_for_price,-0.155,-0.151


,Type of model,after use Feature Selection,number of features,metric_name,score_train,score_valid
0,For classification model,Original features set,376,metric_for_churn,0.971,0.962
1,For classification model,Feature importance,200,metric_for_churn,0.971,0.962
2,For classification model,Permutation importance,120,metric_for_churn,0.968,0.962
3,For classification model,L1 regularization,172,metric_for_churn,0.971,0.962
4,For classification model,Random variable method,159,metric_for_churn,0.971,0.962
5,For classification model,Recursive feature elimination,100,metric_for_churn,0.971,0.962


### Please update the algorithm

In [17]:
def alg1(x):
    """
    Algorithm version 1
    """
    return x['__price_predict']

# column __price_predict must be in train and test dataset
dataset['__priority'] = dataset.apply(alg1, axis=1)

# YOUR WORK IS DONE (Create submission)

In [18]:
# example: 
# top_features_churn = ['max_floor', ...]
# remove duplicated features if exists
top_features_churn = sorted(list(set(top_features_churn)))
dataset['__churn_num_features'] = len(top_features_churn)

# top_features_price = [...]
top_features_price = sorted(list(set(top_features_price)))
dataset['__price_num_features'] = len(top_features_price)

In [22]:
# this columns must be in dataset
prediction_columns = ['__price_predict', '__price_num_features', '__churn_prob', '__churn_num_features', '__priority']
submission = dataset[prediction_columns]

if submission.isnull().sum().sum() > 0:
    raise ValueError(f"Submission dataset contained NaN values")

if submission.shape[0] != SHAPE[0]:
    raise ValueError(f'Incorrect submission file shape. Original {SHAPE[0]}. {submission.shape[0]} are given')

# index must be True
output_path = '4_' + NAME + '.csv'

print(f"output file '{output_path}' was created")
# index must be True
submission.to_csv(output_path, index=True)

output file '4_Kirill_Viksman.csv' was created


In [23]:
fixed = pd.read_csv(f"4_{NAME}.csv")

if "Unnamed: 0" in fixed.columns:
    fixed = fixed.drop(columns=["Unnamed: 0"])

fixed = fixed[
    [
        "__price_predict",
        "__price_num_features",
        "__churn_prob",
        "__churn_num_features",
        "__priority"
    ]
]

fixed.to_csv(f"4_{NAME}.csv", index=False)

check = pd.read_csv(f"4_{NAME}.csv")
print(check.shape)
print(check.columns.tolist())
print(check.isna().sum())
display(check.head())

(30471, 5)
['__price_predict', '__price_num_features', '__churn_prob', '__churn_num_features', '__priority']
__price_predict         0
__price_num_features    0
__churn_prob            0
__churn_num_features    0
__priority              0
dtype: int64


,__price_predict,__price_num_features,__churn_prob,__churn_num_features,__priority
0,4.085270,160,0.000906,120,4.085270
1,7.212441,160,0.118130,120,7.212441
2,1.782924,160,0.760238,120,1.782924
3,3.027526,160,0.151204,120,3.027526
4,4.252799,160,0.011058,120,4.252799
